# 10 — Confidence-Weighted Voting Comparison

Compares the original **majority vote** ensemble with **confidence-weighted voting**
(sum of 6 softmax probability vectors) across all 2,063 test images.

Requires notebooks 01–06 to have been run first.

## Section 0 — Colab / Local Setup

Detects environment, installs packages, and configures Kaggle + HuggingFace credentials.

**Google Colab Secrets required** (Colab → 🔑 Secrets panel):

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | `sk1285` |
| `KAGGLE_KEY` | `7261c6b4046a6bd5c9ba4d1a6f58c98f` |
| `HF_TOKEN` | *(your HuggingFace write token)* |


In [ ]:
import sys, os, json, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(["pip", "install", "kaggle", "huggingface_hub", "-q"], check=False)
    from google.colab import userdata
    _kaggle_user = userdata.get('KAGGLE_USERNAME')
    _kaggle_key  = userdata.get('KAGGLE_KEY')
    HF_TOKEN     = userdata.get('HF_TOKEN')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as _f:
        json.dump({'username': _kaggle_user, 'key': _kaggle_key}, _f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    DATASET_PATH     = "/content/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"
    if not os.path.exists(os.path.join(DATASET_PATH, "Testing")):
        subprocess.run(["kaggle", "datasets", "download",
                        "masoudnickparvar/brain-tumor-mri-dataset",
                        "-p", "/content/"], check=False)
        subprocess.run(["unzip", "-q", "/content/brain-tumor-mri-dataset.zip",
                        "-d", DATASET_PATH], check=False)
        if os.path.exists("/content/brain-tumor-mri-dataset.zip"):
            os.remove("/content/brain-tumor-mri-dataset.zip")
else:
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"
    HF_TOKEN         = os.environ.get('HF_TOKEN', '')

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Environment :", "Colab" if IN_COLAB else "Local")
print("Dataset     :", DATASET_PATH)
print("Models      :", SAVED_MODELS_DIR)


## Section 1 — Imports

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2, joblib, tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42); np.random.seed(42)
print("TF:", tf.__version__)


## Section 2 — Constants

In [ ]:
NOTEBOOK_NAME    = "10_ConfidenceVoting"
HF_REPO_ID       = "shehank98/brain-tumor-mri-models"
CLASS_NAMES      = ["glioma", "meningioma", "notumor", "pituitary"]
IMG_SIZE_CNN     = (224, 224)
IMG_SIZE_PRETRAINED = (299, 299)
RANDOM_SEED      = 42
BATCH_SIZE       = 32

DATASET_PATH     = globals().get("DATASET_PATH",     "../MRI_DATASET/")
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR",      "../results/")
HF_TOKEN         = globals().get("HF_TOKEN",         os.environ.get("HF_TOKEN", ""))

RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)
print("Results dir:", RESULTS_NB_DIR)


## Section 3 — Download & Load All 6 Models

In [ ]:
def _dl(filename):
    """Download from HuggingFace to SAVED_MODELS_DIR using the HF cache."""
    import shutil
    dest = os.path.join(SAVED_MODELS_DIR, filename)
    if os.path.exists(dest):
        print(f"  local: {filename}"); return dest
    print(f"  downloading: {filename} ...", end="", flush=True)
    try:
        from huggingface_hub import hf_hub_download, login as hf_login
        if HF_TOKEN: hf_login(token=HF_TOKEN, add_to_git_credential=False)
        cached = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=f"models/{filename}",
            token=HF_TOKEN or None,
        )
        shutil.copy2(cached, dest)
        print(f" done ({os.path.getsize(dest)/1e6:.1f} MB)")
        return dest
    except Exception as e:
        print(f"
  WARN: {filename}: {e}"); return None

# Only the files actually saved by the standardised training notebooks
needed = [
    "cnn_model.h5", "inceptionv3_model.h5", "xception_model.h5",
    "cnn_ensemble_model.pkl", "inceptionv3_ensemble_model.pkl", "xception_ensemble_model.pkl",
]
print("Checking / downloading models …")
for f in needed:
    _dl(f)


In [ ]:
def _lm(name):
    p = os.path.join(SAVED_MODELS_DIR, name)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")
    m = keras.models.load_model(p); m.trainable = False; return m

def _lpkl(name):
    p = os.path.join(SAVED_MODELS_DIR, name)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")
    return joblib.load(p)

def _make_extractor(base_model):
    """Build a feature extractor from the 'feature_layer' Dense-256 layer."""
    try:
        return keras.Model(inputs=base_model.inputs,
                           outputs=base_model.get_layer('feature_layer').output)
    except Exception:
        for layer in reversed(base_model.layers):
            if len(layer.output_shape) == 2:
                return keras.Model(inputs=base_model.inputs, outputs=layer.output)
    return base_model

print("Loading 3 standalone DL models …")
cnn_model = _lm("cnn_model.h5")
inc_model  = _lm("inceptionv3_model.h5")
xcp_model  = _lm("xception_model.h5")
print("  CNN input:", cnn_model.input_shape,
      " InceptionV3:", inc_model.input_shape,
      " Xception:", xcp_model.input_shape)

# Feature extractors built in memory (no separate *_ensemble.h5 saved by notebooks)
cnn_extractor = _make_extractor(cnn_model)
inc_extractor = _make_extractor(inc_model)
xcp_extractor = _make_extractor(xcp_model)
print("  Feature extractors ready — output shapes:",
      cnn_extractor.output_shape, inc_extractor.output_shape, xcp_extractor.output_shape)

print("Loading 3 classical ensemble models …")
cnn_ens_clf = _lpkl("cnn_ensemble_model.pkl")
inc_ens_clf = _lpkl("inceptionv3_ensemble_model.pkl")
xcp_ens_clf = _lpkl("xception_ensemble_model.pkl")
print("All 6 predictors ready.")


## Section 4 — Load Test Data

Three generators: one for CNN (224×224), one for InceptionV3/Xception (299×299).

In [ ]:
TEST_DIR = os.path.join(DATASET_PATH, "Testing")

datagen = ImageDataGenerator(rescale=1./255)

gen_cnn = datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE,
    class_mode='sparse', classes=CLASS_NAMES, shuffle=False)

gen_299 = datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE_PRETRAINED, batch_size=BATCH_SIZE,
    class_mode='sparse', classes=CLASS_NAMES, shuffle=False)

y_true = gen_cnn.classes
n = len(y_true)
print(f"Test images: {n}")


## Section 5 — Run Inference (All 6 Models)

In [ ]:
print("Running inference (this may take a few minutes) …")

# Standalone DL probabilities
prob_cnn = cnn_model.predict(gen_cnn, verbose=1)     # (N, 4)
prob_inc = inc_model.predict(gen_299, verbose=1)
prob_xcp = xcp_model.predict(gen_299, verbose=1)

# Classical ensemble probabilities via feature extraction
print("Extracting features for classical ensembles …")
feat_cnn = cnn_extractor.predict(gen_cnn, verbose=0)
feat_inc = inc_extractor.predict(gen_299, verbose=0)
feat_xcp = xcp_extractor.predict(gen_299, verbose=0)

prob_cnn_ens = cnn_ens_clf.predict_proba(feat_cnn)    # sklearn returns (N, 4)
prob_inc_ens = inc_ens_clf.predict_proba(feat_inc)
prob_xcp_ens = xcp_ens_clf.predict_proba(feat_xcp)

print("Predictions done.")


## Section 6 — Majority Vote vs Confidence-Weighted Vote

In [ ]:
# --- Majority Vote (original method) ---
from collections import Counter

pred_cnn_cls = np.argmax(prob_cnn, axis=1)
pred_inc_cls = np.argmax(prob_inc, axis=1)
pred_xcp_cls = np.argmax(prob_xcp, axis=1)
pred_cnn_ens_cls = np.argmax(prob_cnn_ens, axis=1)
pred_inc_ens_cls = np.argmax(prob_inc_ens, axis=1)
pred_xcp_ens_cls = np.argmax(prob_xcp_ens, axis=1)

majority_preds = []
for i in range(n):
    votes = [pred_cnn_cls[i], pred_inc_cls[i], pred_xcp_cls[i],
             pred_cnn_ens_cls[i], pred_inc_ens_cls[i], pred_xcp_ens_cls[i]]
    majority_preds.append(Counter(votes).most_common(1)[0][0])
majority_preds = np.array(majority_preds)

# --- Confidence-Weighted Vote (novel extension) ---
combined_probs = prob_cnn + prob_inc + prob_xcp + prob_cnn_ens + prob_inc_ens + prob_xcp_ens
weighted_preds = np.argmax(combined_probs, axis=1)
vote_confidence = combined_probs.max(axis=1) / combined_probs.sum(axis=1)

acc_majority  = accuracy_score(y_true, majority_preds)
acc_weighted  = accuracy_score(y_true, weighted_preds)

print(f"Majority Vote Accuracy  : {acc_majority*100:.2f}%")
print(f"Confidence-Weighted Acc : {acc_weighted*100:.2f}%")
print(f"Improvement             : {(acc_weighted - acc_majority)*100:+.2f}%")


## Section 7 — Classification Reports

In [ ]:
print("\n=== Majority Vote Classification Report ===")
print(classification_report(y_true, majority_preds, target_names=CLASS_NAMES))

print("\n=== Confidence-Weighted Classification Report ===")
print(classification_report(y_true, weighted_preds, target_names=CLASS_NAMES))


## Section 8 — Charts

In [ ]:
import re as _re

# --- Chart 1: Side-by-side confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, preds, title in [
    (axes[0], majority_preds,  f"Majority Vote\n(Acc: {acc_majority*100:.2f}%)"),
    (axes[1], weighted_preds, f"Confidence-Weighted Vote\n(Acc: {acc_weighted*100:.2f}%)"),
]:
    cm = confusion_matrix(y_true, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=[c.capitalize() for c in CLASS_NAMES])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')
plt.suptitle("Voting Strategy Comparison — Confusion Matrices", fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "voting_comparison_confusion_matrix.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)

# --- Chart 2: Per-class F1 improvement ---
from sklearn.metrics import f1_score
f1_maj = f1_score(y_true, majority_preds, average=None, labels=list(range(4)))
f1_wgt = f1_score(y_true, weighted_preds, average=None, labels=list(range(4)))
x = np.arange(len(CLASS_NAMES)); w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, f1_maj, w, label="Majority Vote",           alpha=0.85, color='#3498db')
ax.bar(x + w/2, f1_wgt, w, label="Confidence-Weighted Vote", alpha=0.85, color='#2ecc71')
ax.set_xticks(x); ax.set_xticklabels([c.capitalize() for c in CLASS_NAMES], fontsize=11)
ax.set_ylim(0.80, 1.01); ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("Per-Class F1 Score: Majority Vote vs Confidence-Weighted Vote",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
for i, (a, b) in enumerate(zip(f1_maj, f1_wgt)):
    ax.text(i - w/2, a + 0.002, f"{a:.3f}", ha='center', va='bottom', fontsize=8)
    ax.text(i + w/2, b + 0.002, f"{b:.3f}", ha='center', va='bottom', fontsize=8)
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "voting_comparison_per_class_f1.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)

# --- Chart 3: Confidence distribution ---
fig, ax = plt.subplots(figsize=(10, 4))
colors_conf = ['#e74c3c' if c else '#2ecc71' for c in (weighted_preds != y_true)]
ax.hist(vote_confidence[weighted_preds == y_true],  bins=40, alpha=0.6,
        color='#2ecc71', label='Correct prediction')
ax.hist(vote_confidence[weighted_preds != y_true], bins=40, alpha=0.6,
        color='#e74c3c', label='Misclassified')
ax.set_xlabel("Vote Confidence Score", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Confidence Distribution: Correct vs Misclassified Predictions", fontsize=12, fontweight='bold')
ax.axvline(0.70, linestyle='--', color='gray', lw=1.5, label='Threshold 0.70')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "voting_confidence_distribution.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
print("Saved:", p)


## Section 9 — Save Results CSV

In [ ]:
results_df = pd.DataFrame({
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "majority_pred": [CLASS_NAMES[i] for i in majority_preds],
    "weighted_pred":  [CLASS_NAMES[i] for i in weighted_preds],
    "vote_confidence": vote_confidence,
    "majority_correct": (majority_preds == y_true).astype(int),
    "weighted_correct":  (weighted_preds  == y_true).astype(int),
})
csv_p = os.path.join(RESULTS_NB_DIR, "voting_comparison_results.csv")
results_df.to_csv(csv_p, index=False)
print(f"Saved: {csv_p}")
print(results_df.groupby("true_class")[["majority_correct","weighted_correct"]].mean().round(3))


## Section 10 — Upload to HuggingFace

In [ ]:
def _hf_upload(files, repo_id, token, prefix=""):
    from huggingface_hub import HfApi, login as hf_login
    if not token:
        print("No HF_TOKEN — skipping upload.")
        return
    hf_login(token=token, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
    for p in files:
        if not os.path.exists(p):
            print(f"  skip (missing): {p}"); continue
        rp = (prefix + "/" + os.path.basename(p)).lstrip("/")
        api.upload_file(path_or_fileobj=p, path_in_repo=rp,
                        repo_id=repo_id, repo_type="model")
        print(f"  uploaded: {rp}")
files_to_upload = [
    os.path.join(RESULTS_NB_DIR, "voting_comparison_results.csv"),
    os.path.join(RESULTS_NB_DIR, "voting_comparison_confusion_matrix.jpg"),
    os.path.join(RESULTS_NB_DIR, "voting_comparison_per_class_f1.jpg"),
    os.path.join(RESULTS_NB_DIR, "voting_confidence_distribution.jpg"),
]
_hf_upload(files_to_upload, HF_REPO_ID, HF_TOKEN, prefix=f"results/{NOTEBOOK_NAME}")
print("\nSection 10 complete.")
